<a href="https://colab.research.google.com/github/gndhar/malmo/blob/main/dual-branch-phasenet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content
!git clone https://github.com/gndhar/malmo.git
!pip install zernike

/content
fatal: destination path 'malmo' already exists and is not an empty directory.


In [4]:
%cd /content/malmo/ml
!git pull --rebase

/content/malmo/ml
Already up to date.


In [1]:
%load_ext tensorboard

In [2]:
import torch
import torch.nn as nn

from data_gen import RMDataset
from torch.utils.data import DataLoader, RandomSampler

from zern import ZernikeAberration
from forward import Simulation
from rm import get_Rk_batched
from dual_branch_phasenet import DualBranchPhaseNet, PhaseRetrievalLoss

from torch.optim.lr_scheduler import ReduceLROnPlateau

from tqdm import tqdm

import os
import datetime
from torch.utils.tensorboard import SummaryWriter

import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'data_gen'

In [ ]:
torch.random.manual_seed(42)
import numpy as np
np.random.seed(42)

In [ ]:
logdir = os.path.join("runs", "DualBranchPhaseNet")
writer = SummaryWriter(logdir)

In [ ]:
%tensorboard --logdir=runs

In [22]:
N = 32
zern_n = 4
epochs = 200
device = torch.device("cuda")
save_path = "/content/drive/MyDrive/malmo_models/dual_branch_phasenet.pth"
load_prev = True

In [23]:
train_dataset = RMDataset(
    N=2 * N, size=2048, zern_n=zern_n, seed=42,
    cache_path=f"/content/drive/MyDrive/malmo_cache/train_N{N}.pt",
    num_workers_build=os.cpu_count(),
)
val_dataset = RMDataset(
    N=2 * N, size=256, zern_n=zern_n, seed=420,
    cache_path=f"/content/drive/MyDrive/malmo_cache/val_N{N}.pt",
    num_workers_build=os.cpu_count(),
)

# subset_sampler = RandomSampler(train_dataset, replacement=True, num_samples=1024)
train_dataloader = DataLoader(train_dataset, batch_size=32, num_workers=os.cpu_count(), pin_memory=True, persistent_workers=True)

val_dataset = RMDataset(N=2 * N, size=256, zern_n=zern_n, seed=420)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=os.cpu_count(), pin_memory=True, persistent_workers=True)

Loading cached objects from /content/drive/MyDrive/malmo_cache/train_N32.pt
Loading cached objects from /content/drive/MyDrive/malmo_cache/val_N32.pt


In [12]:
zern_gen = ZernikeAberration(N=N, zern_n=zern_n).to(device)
simulation = Simulation(N, dtype=torch.complex64).to(device)


def center_crop(x: torch.Tensor, size: int) -> torch.Tensor:
    """Center-crop the last two spatial dims of x down to (size, size)."""
    h, w = x.shape[-2], x.shape[-1]
    top = (h - size) // 2
    left = (w - size) // 2
    return x[..., top:top + size, left:left + size]


# Aperture mask derived from the actual (cropped) aberration field rather than
# recomputed geometrically -- avoids any off-by-one mismatch between the
# model's mask and the data pipeline's true aperture.
with torch.no_grad():
    c_in_sample, _, _ = next(iter(train_dataloader))
    ab_sample = zern_gen(c_in_sample.to(device))
    ab_sample_cropped = center_crop(ab_sample, N)
    aperture_mask = (ab_sample_cropped[0].abs() > 1e-6).float()
# plt.imshow(aperture_mask.cpu().numpy(), cmap="gray")
model = DualBranchPhaseNet(N=N, embed_dim=256, aperture_mask=aperture_mask).to(device)
if load_prev:
    model.load_state_dict(torch.load(save_path))

In [24]:
print(f"Model param count: {sum(p.numel() for p in model.parameters()):,}")

Model param count: 29,079,430


In [25]:
c_in, c_out, obj = next(iter(train_dataloader))
c_in, c_out, obj = c_in.to(device), c_out.to(device), obj.to(device)

with torch.no_grad():
    ab_in = zern_gen(c_in)
    ab_out = zern_gen(c_out)
    k_outs = simulation(ab_in, ab_out, obj)
    dummy_Rk = get_Rk_batched(k_in=simulation.k_in_cropped, k_outs=k_outs, N=N)

writer.add_graph(model, dummy_Rk)

writer.flush()

/content/malmo/ml/dual_branch_phasenet.py:137: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if x.shape[-2:] != (self.N, self.N):


In [26]:
criterion = PhaseRetrievalLoss(alpha_phase=1.0, alpha_kernel=0.0)


def compute_object_kernel(obj: torch.Tensor) -> torch.Tensor:
    """Ground-truth object kernel for PhaseRetrievalLoss.

    Uses the same k-space/real-space convention as the rest of the codebase:
        img_raw = ifftshift(ifft2(fftshift(image_k))) * 4.0
    so the forward direction (obj -> image_k) is:
        image_k = ifftshift(fft2(fftshift(obj))) / 4.0

    obj is (2N, 2N); o_pred from KernelProjectionHead is (2N-1, 2N-1), so we
    drop the 0th index along each spatial axis after computing image_k to
    land on a symmetric (2N-1, 2N-1) grid.

    NOTE: cross-check this convention against how o_pred's coordinate grid
    is actually defined before trusting the loss values.
    """
    obj_c = obj if torch.is_complex(obj) else obj.to(torch.complex64)
    shifted = torch.fft.fftshift(obj_c, dim=(-2, -1))
    Ok = torch.fft.fft2(shifted, dim=(-2, -1)) / 4.0
    Ok = torch.fft.ifftshift(Ok, dim=(-2, -1))
    Ok = Ok[..., 1:, 1:]
    return Ok

In [18]:
def run_epoch(dataloader, is_train, epoch):
    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0

    with torch.set_grad_enabled(is_train):
        for c_in, c_out, obj in dataloader:
            c_in, c_out, obj = c_in.to(device), c_out.to(device), obj.to(device)

            with torch.no_grad():
                ab_in = zern_gen(c_in)
                ab_out = zern_gen(c_out)
                k_outs = simulation(ab_in, ab_out, obj)
                Rk = get_Rk_batched(k_in=simulation.k_in_cropped, k_outs=k_outs, N=N)
                p_target = center_crop(ab_in, N)
                q_target = center_crop(ab_out, N)
                o_target = compute_object_kernel(obj)

            if is_train:
                optimizer.zero_grad()

            p_pred, q_pred, o_pred = model(Rk)
            loss = criterion(p_pred, q_pred, o_pred, p_target, q_target, o_target)

            if is_train:
                loss.backward()

                # for name, param in model.named_parameters():
                #     if param.grad is not None:
                #         writer.add_histogram(f'Gradients/{name}', param.grad, epoch)

                optimizer.step()

            total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)

    phase = "train" if is_train else "val"
    writer.add_scalar(f'Loss/{phase}', avg_loss, epoch)

    return avg_loss

In [ ]:
optimizer = torch.optim.AdamW(params=model.parameters(), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)

In [ ]:
print("--- Starting Training ---")

for epoch in tqdm(range(epochs)):
    writer.add_scalar('LR/main', optimizer.param_groups[0]['lr'], epoch)
    train_loss = run_epoch(train_dataloader, is_train=True, epoch=epoch)
    val_loss = run_epoch(val_dataloader, is_train=False, epoch=epoch)
    scheduler.step(val_loss)
    print(
        f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}"
    )
    writer.flush()

--- Starting Training ---


  0%|          | 1/200 [01:10<3:54:14, 70.62s/it]

Epoch [1/200] | Train Loss: 0.8177 | Val Loss: 0.7990


  1%|          | 2/200 [02:26<4:03:07, 73.67s/it]

Epoch [2/200] | Train Loss: 0.7923 | Val Loss: 0.8130


...

Epoch [34/200] | Train Loss: 0.4950 | Val Loss: 0.4962


 18%|█▊        | 35/200 [44:30<3:30:04, 76.39s/it]

Epoch [35/200] | Train Loss: 0.4968 | Val Loss: 0.5007


 18%|█▊        | 36/200 [45:46<3:28:38, 76.33s/it]

Epoch [36/200] | Train Loss: 0.4947 | Val Loss: 0.4484


In [ ]:
save_path = "/content/drive/MyDrive/malmo_models/dual_branch_phasenet.pth"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
torch.save(model.state_dict(), save_path)